# Post-Hoc Calibration Map Ablation

Compares calibration methods (Platt Scaling, Isotonic, Histogram Binning) applied to signal-space confidence (Linguistic confidence, Token probability, Semantic uncertainty) across all datasets and models. Train: first 30%, Test: last 70%. Reports generalised ECE and FD on the test split.

In [1]:
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
sys.path.insert(0, '/home/ivan/lm-confidence-evaluation-harness')

warnings.filterwarnings('ignore')

/home/ivan/miniconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from lm_conf.default_utils.custom_types import OrganisedOutputs
from lm_conf.post_processing.metrics import generalised_ece, faithfulness_divergence
from calibration.utils import (
    in_domain_numerical_post_hoc_calibration,
    _print_signal_metrics_from_lists,
)

## Helper Functions

In [3]:
COMMON_DIR = '/hdd/ivny'
DATASETS = ['mmlu', 'squadv2', 'truthful_qa']
SIGNALS = {
    'lc': 'Linguistic confidence',
    'tp': 'Token probability',
    'su': 'Semantic uncertainty',
}
METHODS = ['platt_uni', 'isotonic', 'hist_bin', 'temperature_scaling']
METHOD_LABELS = {
    'platt_uni':           'Platt Scaling',
    'isotonic':            'Isotonic',
    'hist_bin':            'Histogram Binning',
    'temperature_scaling': 'Temperature Scaling',
}
DATASET_LABELS = {
    'mmlu': 'MMLU',
    'squadv2': 'SQuAD 2.0',
    'truthful_qa': 'TruthfulQA',
}


def _leaf_dirs(root_dir):
    """Return all leaf directories (no sub-dirs) under root_dir, sorted."""
    leaf_dirs = []
    for root, dirs, files in os.walk(root_dir):
        if not dirs:
            leaf_dirs.append(root)
    return sorted(leaf_dirs)


def _get_latest_leaf(parent_dir: str) -> str | None:
    """
    If parent_dir has multiple child dirs (siblings), return the one with the
    most recent mtime; otherwise return the single child. Returns None if
    graded_outputs_0.pkl is missing in the chosen dir.
    """
    try:
        subdirs = [
            os.path.join(parent_dir, d)
            for d in os.listdir(parent_dir)
            if os.path.isdir(os.path.join(parent_dir, d))
        ]
    except FileNotFoundError:
        return None
    if not subdirs:
        return None
    chosen = max(subdirs, key=os.path.getmtime)
    pkl = os.path.join(chosen, 'graded_outputs_0.pkl')
    return chosen if os.path.exists(pkl) else None


def load_pkl(path: str, filename: str):
    with open(os.path.join(path, filename), 'rb') as f:
        return pickle.load(f)


def compute_metrics(confidences, accuracies, answers):
    """
    Compute generalised ECE and mean FD from lists of BetaDistribution objects.
    Filters out None confidences.
    """
    valid = [
        (c, a, ans)
        for c, a, ans in zip(confidences, accuracies, answers)
        if c is not None
    ]
    if not valid:
        return float('nan'), float('nan'), 0
    confs, accs, anss = zip(*valid)
    org = OrganisedOutputs(
        accuracy_scores=[list(accs)],
        extracted_confidences=[list(confs)],
        extracted_answers=[list(anss)],
    )
    ece = generalised_ece({}, org)[0]
    fd  = faithfulness_divergence({}, org)[0]
    return float(ece), float(fd), len(valid)

## Discover Runs

Walk `/hdd/ivny/results/{dataset}/direct_qa_{signal}/{org}/{model}/` and select the latest timestamp dir per model.

In [4]:
runs = []  # list of dicts: dataset, signal, model, leaf_dir

for dataset in DATASETS:
    results_root = os.path.join(COMMON_DIR, 'results', dataset)
    dqa_dirs = sorted([
        d for d in os.listdir(results_root)
        if d.startswith('direct_qa_')
    ])
    for dqa in dqa_dirs:
        # Extract signal suffix e.g. 'direct_qa_unified_lc' -> 'lc'
        suffix = dqa.replace('direct_qa_unified_', '').replace('direct_qa_', '')
        if suffix not in SIGNALS:
            continue
        dqa_path = os.path.join(results_root, dqa)
        # Walk org/model levels
        for org in os.listdir(dqa_path):
            org_path = os.path.join(dqa_path, org)
            if not os.path.isdir(org_path):
                continue
            for model in os.listdir(org_path):
                model_path = os.path.join(org_path, model)
                if not os.path.isdir(model_path):
                    continue
                leaf = _get_latest_leaf(model_path)
                if leaf is None:
                    continue
                runs.append({
                    'dataset': dataset,
                    'signal':  suffix,
                    'model':   f'{org}/{model}',
                    'leaf':    leaf,
                })

print(f'Total runs found: {len(runs)}')
pd.DataFrame(runs)[['dataset', 'signal', 'model']].value_counts().reset_index()

Total runs found: 63


,dataset,signal,model,count
0,mmlu,lc,mistralai/Mistral-7B-Instruct-v0.3,1
1,mmlu,lc,google/gemma-4-31B-it,1
2,mmlu,lc,meta-llama/Llama-3.1-8B-Instruct,1
3,mmlu,lc,openai/gpt-oss-20b,1
4,mmlu,lc,openai/gpt-oss-120b,1
...,...,...,...,...
58,truthful_qa,tp,meta-llama/Llama-3.1-8B-Instruct,1
59,truthful_qa,tp,openai/gpt-oss-20b,1
60,truthful_qa,tp,openai/gpt-oss-120b,1
61,truthful_qa,tp,qwen/Qwen3-235B-A22B-Instruct-2507-tput,1


## Run Calibration Ablation

For each run, load `graded_outputs_0.pkl`, apply each calibration method (first 30% train, last 70% test), and record ECE and FD on the test portion.

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _valid_acc(a):
    if a is None or a == '':
        return False
    try:
        v = float(a)
        return not np.isnan(v) and v in (0.0, 1.0)
    except (ValueError, TypeError):
        return False


def process_run(run):
    try:
        graded: OrganisedOutputs = load_pkl(run['leaf'], 'graded_outputs_0.pkl')
    except Exception:
        return []

    raw_confs = graded.extracted_confidences[0]
    raw_accs  = graded.accuracy_scores[0]
    raw_ans   = graded.extracted_answers[0]

    valid_mask = [c is not None and _valid_acc(a) for c, a in zip(raw_confs, raw_accs)]
    confs = [c for c, v in zip(raw_confs, valid_mask) if v]
    accs  = [float(a) for a, v in zip(raw_accs, valid_mask) if v]
    ans   = [a for a, v in zip(raw_ans, valid_mask) if v]
    if len(confs) < 10:
        return []

    n_train = max(1, int(np.ceil(0.30 * len(confs))))
    rows = []

    pre_ece, pre_fd, n_test = compute_metrics(confs[n_train:], accs[n_train:], ans[n_train:])
    rows.append({'dataset': run['dataset'], 'signal': run['signal'], 'model': run['model'],
                 'method': 'uncalibrated', 'ECE': pre_ece, 'FD': pre_fd, 'n_test': n_test})

    confs_arr = np.array(confs, dtype=object)
    accs_arr  = np.array(accs, dtype=float)

    for method in METHODS:
        try:
            cal_confs = in_domain_numerical_post_hoc_calibration(confs_arr, accs_arr, method=method)
            post_ece, post_fd, n = compute_metrics(cal_confs[n_train:], accs[n_train:], ans[n_train:])
        except Exception:
            post_ece, post_fd, n = float('nan'), float('nan'), 0
        rows.append({'dataset': run['dataset'], 'signal': run['signal'], 'model': run['model'],
                     'method': method, 'ECE': post_ece, 'FD': post_fd, 'n_test': n})
    return rows


records = []
with ThreadPoolExecutor(max_workers=None) as pool:
    futures = {pool.submit(process_run, run): run for run in runs}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Calibrating'):
        records.extend(fut.result())

df_records = pd.DataFrame(records)
print(f'Total records: {len(df_records)}')
df_records.head()

Calibrating:   0%|          | 0/63 [00:00<?, ?it/s]

[hist_bin] bin_accuracies=[0.47, 0.481, 0.483, 0.486, 0.49, 0.495, 0.499, 0.503, 0.507, 0.511, 0.514, 0.514, 0.516, 0.521, 0.524, 0.525, 0.525, 0.525, 0.525, 0.531]
[hist_bin] bin_accuracies=[0.005, 0.038, 0.082, 0.114, 0.132, 0.142, 0.149, 0.155, 0.161, 0.168, 0.174, 0.179, 0.185, 0.196, 0.224, 0.294, 0.414, 0.534, 0.611, 0.649]
[hist_bin] bin_accuracies=[0.485, 0.485, 0.485, 0.485, 0.485, 0.485, 0.485, 0.485, 0.488, 0.497, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.528]
[hist_bin] bin_accuracies=[0.089, 0.112, 0.153, 0.189, 0.211, 0.222, 0.23, 0.239, 0.25, 0.262, 0.274, 0.286, 0.301, 0.32, 0.352, 0.412, 0.507, 0.604, 0.665, 0.693]
[hist_bin] bin_accuracies=[0.005, 0.023, 0.062, 0.119, 0.187, 0.252, 0.308, 0.352, 0.385, 0.41, 0.431, 0.45, 0.47, 0.5, 0.554, 0.631, 0.697, 0.733, 0.75, 0.756]
[hist_bin] bin_accuracies=[0.707, 0.707, 0.731, 0.75, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754, 0.754]
[hist_bin] bin_accu

Calibrating:  51%|█████     | 32/63 [21:07<00:17,  1.81it/s]    

[hist_bin] bin_accuracies=[0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.445, 0.492, 0.553, 0.639, 0.696]
[hist_bin] bin_accuracies=[0.535, 0.535, 0.535, 0.535, 0.535, 0.535, 0.535, 0.535, 0.535, 0.535, 0.535, 0.535, 0.535, 0.562, 0.566, 0.594, 0.636, 0.684, 0.706, 0.766]
[hist_bin] bin_accuracies=[0.025, 0.075, 0.125, 0.175, 0.225, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809, 0.809]
[hist_bin] bin_accuracies=[0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.626, 0.649, 0.693, 0.709]
[hist_bin] bin_accuracies=[0.001, 0.01, 0.034, 0.081, 0.161, 0.28, 0.386, 0.405, 0.405, 0.405, 0.412, 0.438, 0.464, 0.485, 0.516, 0.579, 0.648, 0.684, 0.684, 0.721]
[hist_bin] bin_accuracies=[0.235, 0.29, 0.29, 0.29, 0.303, 0.321, 0.321, 0.321, 0.321, 0.321, 0.321, 0.338, 0.371, 0.426, 0.426, 0.468, 0.468, 0.468, 0.591, 0.612]
[hi

Calibrating:  84%|████████▍ | 53/63 [29:37<00:06,  1.60it/s]   

[temperature_scaling] T*=4.0153
[temperature_scaling] T*=7.2355
[temperature_scaling] T*=6.6540
[temperature_scaling] T*=9.6277
[temperature_scaling] T*=46.9296
[temperature_scaling] T*=9.8909
[temperature_scaling] T*=4.4525
[temperature_scaling] T*=17.2315
[temperature_scaling] T*=7.0654
[temperature_scaling] T*=2.7016


Calibrating: 100%|██████████| 63/63 [31:07<00:00, 29.64s/it]

Total records: 315


,dataset,signal,model,method,ECE,FD,n_test
0,squadv2,lc,openai/gpt-oss-120b,uncalibrated,0.378480,2.065489,8275
1,squadv2,lc,openai/gpt-oss-120b,platt_uni,0.136115,0.455690,8275
2,squadv2,lc,openai/gpt-oss-120b,isotonic,0.137528,0.472321,8275
3,squadv2,lc,openai/gpt-oss-120b,hist_bin,0.134497,0.453830,8275
4,squadv2,lc,openai/gpt-oss-120b,temperature_scaling,0.136689,0.454816,8275


## Aggregate Results

Mean ECE and FD across all (dataset, model) combinations, grouped by (signal, method).

In [6]:
METHOD_ORDER = ['uncalibrated'] + METHODS
SIGNAL_ORDER = ['lc', 'tp', 'su']

# Aggregate: mean ECE and FD over all (dataset, model) per (signal, method)
agg = (
    df_records
    .groupby(['signal', 'method'], sort=False)[['ECE', 'FD']]
    .agg(['mean', 'std'])
    .round(4)
)
agg.columns = ['ECE_mean', 'ECE_std', 'FD_mean', 'FD_std']
agg = agg.reset_index()

# Pivot: rows = method, columns = (signal, metric)
pivot_rows = []
for method in METHOD_ORDER:
    row = {'Method': METHOD_LABELS.get(method, 'Uncalibrated')}
    for sig in SIGNAL_ORDER:
        sub = agg[(agg['signal'] == sig) & (agg['method'] == method)]
        if sub.empty:
            row[f'{SIGNALS[sig]}_ECE'] = '-'
            row[f'{SIGNALS[sig]}_FD']  = '-'
        else:
            ece_m = sub['ECE_mean'].values[0]
            ece_s = sub['ECE_std'].values[0]
            fd_m  = sub['FD_mean'].values[0]
            fd_s  = sub['FD_std'].values[0]
            row[f'{SIGNALS[sig]}_ECE'] = f'{ece_m:.4f} ± {ece_s:.4f}'
            row[f'{SIGNALS[sig]}_FD']  = f'{fd_m:.4f} ± {fd_s:.4f}'
    pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).set_index('Method')
pivot_df.columns = pd.MultiIndex.from_tuples(
    [(SIGNALS[sig], metric) for sig in SIGNAL_ORDER for metric in ['ECE', 'FD']]
)
print('Aggregated results (mean ± std across all datasets and models):')
pivot_df

Aggregated results (mean ± std across all datasets and models):


Linguistic confidence                  Token probability  \
                                      ECE               FD               ECE   
Method                                                                         
Uncalibrated              0.2802 ± 0.1067  1.4871 ± 0.6278   0.2615 ± 0.1110   
Platt Scaling             0.1085 ± 0.0177  0.4862 ± 0.0421   0.0848 ± 0.0318   
Isotonic                  0.1144 ± 0.0193  0.6847 ± 0.5298   0.0902 ± 0.0341   
Histogram Binning         0.1099 ± 0.0191  0.4875 ± 0.0416   0.0897 ± 0.0393   
Temperature Scaling       0.1267 ± 0.0224  0.5465 ± 0.1479   0.1020 ± 0.0317   

                                       Semantic uncertainty                   
                                    FD                  ECE               FD  
Method                                                                        
Uncalibrated         68.8792 ± 96.5199      0.2669 ± 0.1135  2.1118 ± 0.7619  
Platt Scaling          0.5002 ± 0.0603      0.0595 ± 0.0140  0.5063 ± 0.0506  
Isotonic               1.6457 ± 1.4949      0.0650 ± 0.0169  0.7177 ± 0.4387  
Histogram Binning      0.5139 ± 0.1037      0.0603 ± 0.0170  0.5024 ± 0.0472  
Temperature Scaling    0.6463 ± 0.3798      0.0796 ± 0.0306  0.5195 ± 0.0561

## Per-Signal Summary Table (means only)

In [7]:
# Clean numeric table for visual inspection
summary_rows = []
for method in METHOD_ORDER:
    row = {'Method': METHOD_LABELS.get(method, 'Uncalibrated')}
    for sig in SIGNAL_ORDER:
        sub = agg[(agg['signal'] == sig) & (agg['method'] == method)]
        if sub.empty:
            row[f'{SIGNALS[sig]} ECE'] = float('nan')
            row[f'{SIGNALS[sig]} FD']  = float('nan')
        else:
            row[f'{SIGNALS[sig]} ECE'] = round(float(sub['ECE_mean'].values[0]), 4)
            row[f'{SIGNALS[sig]} FD']  = round(float(sub['FD_mean'].values[0]), 4)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Method')

def highlight_best(col):
    """Highlight the row with the lowest value in each column."""
    is_best = col == col.min()
    return ['font-weight: bold; color: #1a7a1a' if v else '' for v in is_best]

display(summary_df.style.apply(highlight_best, axis=0).format('{:.4f}'))

,Linguistic confidence ECE,Linguistic confidence FD,Token probability ECE,Token probability FD,Semantic uncertainty ECE,Semantic uncertainty FD
Method,,,,,,
Uncalibrated,0.2802,1.4871,0.2615,68.8792,0.2669,2.1118
Platt Scaling,0.1085,0.4862,0.0848,0.5002,0.0595,0.5063
Isotonic,0.1144,0.6847,0.0902,1.6457,0.0650,0.7177
Histogram Binning,0.1099,0.4875,0.0897,0.5139,0.0603,0.5024
Temperature Scaling,0.1267,0.5465,0.1020,0.6463,0.0796,0.5195


## Per-Dataset Breakdown

In [8]:
for dataset in DATASETS:
    ds_df = df_records[df_records['dataset'] == dataset]
    ds_agg = (
        ds_df.groupby(['signal', 'method'])[['ECE', 'FD']]
        .mean().round(4).reset_index()
    )

    rows = []
    for method in METHOD_ORDER:
        row = {'Method': METHOD_LABELS.get(method, 'Uncalibrated')}
        for sig in SIGNAL_ORDER:
            sub = ds_agg[(ds_agg['signal'] == sig) & (ds_agg['method'] == method)]
            if sub.empty:
                row[f'{SIGNALS[sig]} ECE'] = float('nan')
                row[f'{SIGNALS[sig]} FD']  = float('nan')
            else:
                row[f'{SIGNALS[sig]} ECE'] = round(float(sub['ECE'].values[0]), 4)
                row[f'{SIGNALS[sig]} FD']  = round(float(sub['FD'].values[0]), 4)
        rows.append(row)

    tbl = pd.DataFrame(rows).set_index('Method')
    print(f'\n=== {DATASET_LABELS[dataset]} ===')
    display(tbl.style.apply(highlight_best, axis=0).format('{:.4f}'))


=== MMLU ===


,Linguistic confidence ECE,Linguistic confidence FD,Token probability ECE,Token probability FD,Semantic uncertainty ECE,Semantic uncertainty FD
Method,,,,,,
Uncalibrated,0.1665,0.7415,0.2133,132.7099,0.1510,1.3551
Platt Scaling,0.1113,0.5173,0.0889,0.5501,0.0556,0.5518
Isotonic,0.1109,0.5369,0.0905,1.0575,0.0547,0.5827
Histogram Binning,0.1103,0.5142,0.0870,0.5482,0.0530,0.5414
Temperature Scaling,0.1351,0.6807,0.0992,0.9404,0.0577,0.5683



=== SQuAD 2.0 ===


,Linguistic confidence ECE,Linguistic confidence FD,Token probability ECE,Token probability FD,Semantic uncertainty ECE,Semantic uncertainty FD
Method,,,,,,
Uncalibrated,0.3286,1.9453,0.3199,57.2981,0.3431,2.6951
Platt Scaling,0.1086,0.4699,0.0762,0.4664,0.0584,0.4910
Isotonic,0.1106,0.4930,0.0790,0.5081,0.0677,0.6069
Histogram Binning,0.1145,0.4778,0.0811,0.4687,0.0582,0.4801
Temperature Scaling,0.1145,0.4706,0.1025,0.4837,0.0840,0.4963



=== TruthfulQA ===


,Linguistic confidence ECE,Linguistic confidence FD,Token probability ECE,Token probability FD,Semantic uncertainty ECE,Semantic uncertainty FD
Method,,,,,,
Uncalibrated,0.3455,1.7745,0.2514,16.6295,0.3066,2.2853
Platt Scaling,0.1056,0.4715,0.0895,0.4841,0.0646,0.4761
Isotonic,0.1217,1.0242,0.1011,3.3714,0.0725,0.9635
Histogram Binning,0.1049,0.4706,0.1010,0.5248,0.0698,0.4857
Temperature Scaling,0.1304,0.4882,0.1042,0.5148,0.0972,0.4938


## Delta Table (post − pre calibration)

In [9]:
# Compute delta: calibrated method minus uncalibrated, for each (dataset, model, signal)
df_uncal = df_records[df_records['method'] == 'uncalibrated'].set_index(
    ['dataset', 'signal', 'model']
)[['ECE', 'FD']].rename(columns={'ECE': 'ECE_base', 'FD': 'FD_base'})

df_cal = df_records[df_records['method'] != 'uncalibrated'].copy()
df_cal = df_cal.join(df_uncal, on=['dataset', 'signal', 'model'])
df_cal['ΔECE'] = df_cal['ECE'] - df_cal['ECE_base']
df_cal['ΔFD']  = df_cal['FD']  - df_cal['FD_base']

delta_agg = (
    df_cal.groupby(['signal', 'method'])[['ΔECE', 'ΔFD']]
    .mean().round(4).reset_index()
)

delta_rows = []
for method in METHODS:
    row = {'Method': METHOD_LABELS[method]}
    for sig in SIGNAL_ORDER:
        sub = delta_agg[(delta_agg['signal'] == sig) & (delta_agg['method'] == method)]
        if sub.empty:
            row[f'{SIGNALS[sig]} ΔECE'] = float('nan')
            row[f'{SIGNALS[sig]} ΔFD']  = float('nan')
        else:
            row[f'{SIGNALS[sig]} ΔECE'] = round(float(sub['ΔECE'].values[0]), 4)
            row[f'{SIGNALS[sig]} ΔFD']  = round(float(sub['ΔFD'].values[0]),  4)
    delta_rows.append(row)

delta_df = pd.DataFrame(delta_rows).set_index('Method')
print('Δ = calibrated − uncalibrated (negative = improvement)')

def highlight_delta(col):
    """Green for the most negative (biggest improvement), red for most positive."""
    styles = []
    for v in col:
        if v == col.min():
            styles.append('font-weight: bold; color: #1a7a1a')
        elif v == col.max():
            styles.append('color: #a00')
        else:
            styles.append('')
    return styles

display(delta_df.style.apply(highlight_delta, axis=0).format('{:+.4f}'))

Δ = calibrated − uncalibrated (negative = improvement)


,Linguistic confidence ΔECE,Linguistic confidence ΔFD,Token probability ΔECE,Token probability ΔFD,Semantic uncertainty ΔECE,Semantic uncertainty ΔFD
Method,,,,,,
Platt Scaling,-0.1717,-1.0009,-0.1767,-68.3790,-0.2073,-1.6055
Isotonic,-0.1658,-0.8024,-0.1713,-67.2335,-0.2019,-1.3941
Histogram Binning,-0.1703,-0.9996,-0.1718,-68.3653,-0.2065,-1.6094
Temperature Scaling,-0.1536,-0.9406,-0.1596,-68.2329,-0.1872,-1.5923
